# Shape toroidal AutoEncoder
latent space $\mathbb{R}^D / {A\mathbb{Z}^D$ with $A$ learnable

In [80]:
import os
import sys

mvae_dir = os.path.split(os.getcwd())[0]
if mvae_dir not in sys.path:
    sys.path.append(mvae_dir)

In [81]:
%pwd
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Imports

In [82]:
import torch
import numpy as np
import torch.optim as optim

import lib.dataloaders as dataloaders
from lib.models import ShapeToroidalAE
from lib.trainer import AETrainer
import lib.utils as utils
import lib.models.utils.save_load_models as modelutils



### Set up and initialize data loader

In [83]:
# Set random seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

### Dataloader

In [89]:
from torch.utils.data import TensorDataset, DataLoader, random_split

batch_size = 128

def generate_shaped_torus_pointcloud(A, num_points=80000, noise_std=0.00, device='cpu'):
    d = A.shape[0]  # Latent dimension (e.g., 2)
    embed_dim = 5  # Target embedding dimension

    # Step 1: Sample θ ∈ [0,1]^d uniformly
    theta = torch.rand(num_points, d, device=device)

    # Step 2: Compute standard torus embedding (4D)
    cos_theta = torch.cos(2 * torch.pi * theta)
    sin_theta = torch.sin(2 * torch.pi * theta)

    # Step 3: Apply shaping matrix A
    cos_shaped = cos_theta @ A.T  # [num_points, d]
    sin_shaped = sin_theta @ A.T  # [num_points, d]

    # Step 4: Form the shaped torus in 4D
    pointcloud_4D = torch.cat([cos_shaped, sin_shaped], dim=1)  # [num_points, 2d]

    # Step 5: Add optional Gaussian noise
    if noise_std > 0:
        noise = noise_std * torch.randn_like(pointcloud_4D)
        pointcloud_4D += noise

    # Step 6: Embed into 10D (pad with zeros)
    extra_dims = torch.zeros(num_points, embed_dim - 4, device=device)
    pointcloud_10D = torch.cat([pointcloud_4D, extra_dims], dim=1)  # [num_points, 10]

    # Step 7: Generate a random translation vector t ∈ R^10
    #translation = torch.randn(embed_dim, device=device)  # Shape: [10]
    translation = torch.zeros(embed_dim, device=device)

    # Step 8: Apply translation
    pointcloud_10D_translated = pointcloud_10D + translation  # [num_points, 10]

    return pointcloud_10D_translated, theta  # Return theta for validation



A = torch.tensor([[1.0, 0.0], [0.0, 1.0]])  # Example A
pointcloud, theta_gt = generate_shaped_torus_pointcloud(A, noise_std=0.0)

# Wrap pointcloud in dataset
dataset = TensorDataset(pointcloud)

# Split dataset (80% train, 20% test)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Create DataLoader
dataloader = train_loader, test_loader

### Model

In [90]:
latent_dim = 2
data_dim = 5

model = ShapeToroidalAE(data_dim, latent_dim)

### Optimizer

In [91]:
learning_rate = 0.001

optimizer = optim.Adam(model.parameters(), lr=learning_rate)

### Train and evaluate model

In [92]:
num_epochs = 5
log_interval = 100
device = "cpu"

trainer_config = {'num_epochs': num_epochs, 'log_interval': log_interval, 'device': device}

ae_history = AETrainer(model, dataloader, optimizer, trainer_config).train()

Trainer successfully initialized.
Start training...
Epoch 0
Step [100/500], Loss: 0.4233, Shape matrix A:1, A_inv_T:1
Step [200/500], Loss: 0.3978, Shape matrix A:1, A_inv_T:1
Step [300/500], Loss: 0.3486, Shape matrix A:1, A_inv_T:1
Step [400/500], Loss: 0.3353, Shape matrix A:1, A_inv_T:1
Step [500/500], Loss: 0.3017, Shape matrix A:1, A_inv_T:1
Epoch 1/5, Train Loss: 0.3826, Test Loss: 0.3132
Epoch 1
Step [100/500], Loss: 0.3077, Shape matrix A:1, A_inv_T:1
Step [200/500], Loss: 0.3093, Shape matrix A:1, A_inv_T:1
Step [300/500], Loss: 0.2984, Shape matrix A:1, A_inv_T:1
Step [400/500], Loss: 0.3022, Shape matrix A:1, A_inv_T:1
Step [500/500], Loss: 0.2928, Shape matrix A:1, A_inv_T:1
Epoch 2/5, Train Loss: 0.3031, Test Loss: 0.3004
Epoch 2
Step [100/500], Loss: 0.2996, Shape matrix A:1, A_inv_T:1
Step [200/500], Loss: 0.2314, Shape matrix A:1, A_inv_T:1
Step [300/500], Loss: 0.2233, Shape matrix A:1, A_inv_T:1
Step [400/500], Loss: 0.1998, Shape matrix A:1, A_inv_T:1
Step [500/500]